# Create a Bedrock knowledge base with kbforge

This notebook takes you from **documents in S3 (or a website)** to a **working Amazon Bedrock knowledge base**.
Nothing is created in AWS until you run `kb.apply()` in step 5. Every cell before that only *looks*.

**Short on time?** Run sections 1, 4, 5, 6 and 7 and keep the defaults. Section 3 walks through every setting you can change, with an example of each.

| | Section | What happens |
|---|---|---|
| 1 | [Install](#1.-Install) | Installs kbforge |
| 2 | [Check your setup](#2.-Check-your-setup) | Shows your region and what you need |
| 3 | [Choose your settings](#3.-Choose-your-settings) | Every option with an example. Nothing touches AWS |
| 4 | [Describe your knowledge base](#4.-Describe-your-knowledge-base) | Your actual definition (or load it from a YAML/JSON file) |
| 5 | [Review the plan](#5.-Review-the-plan) | Read-only checks plus a list of everything it would create |
| 6 | [Build it](#6.-Build-it) | Creates everything, with live progress |
| 7 | [Try it](#7.-Try-it) | Search it and use the results in code |
| 8 | [Keep it up to date](#8.-Keep-it-up-to-date) | Re-sync after documents change |
| 9 | [Share the definition](#9.-Share-the-definition) | Save as YAML/JSON and use the `kbforge` command |
| 10 | [Compare settings](#10.-Compare-settings) | Try a second chunking strategy side by side |
| 11 | [Clean up](#11.-Clean-up) | Delete what kbforge created |
| | [Troubleshooting](#Troubleshooting) | Common messages and what to do |

Full guide with screenshots: https://utkarsh5026.github.io/bedrcok/

## 1. Install

Run once per kernel. Restart the kernel afterwards if `pip` asks you to.

- `kbforge[notebook]` adds the interactive form (`kbforge.wizard()`).
- If you plan to use **OpenSearch Serverless** as the vector store, install `kbforge[all]` instead, which adds the OpenSearch client.

In [ ]:
%pip install -q "kbforge[notebook] @ git+https://github.com/utkarsh5026/bedrcok.git"
# OpenSearch Serverless as well:
# %pip install -q "kbforge[all] @ git+https://github.com/utkarsh5026/bedrcok.git"

## 2. Check your setup

The card below shows the region your notebook will build in and what you need before you start:

- Documents in S3 **in the same region** as the knowledge base, or a website you're allowed to crawl.
- A notebook role allowed to use Bedrock, S3 Vectors and IAM. The exact permissions are in [`docs/notebook-role-policy.json`](https://github.com/utkarsh5026/bedrcok/blob/main/docs/notebook-role-policy.json). If you can't create IAM roles, ask an admin for one and pass it as `role_arn=` (see 3.5).
- **Model access** enabled for your embedding model, in the Bedrock console under *Model access*.

In [ ]:
import kbforge

print("kbforge", kbforge.__version__)
kbforge.guide()

## 3. Choose your settings

Only two things are required: a **name** and a **source**. Everything else has a sensible default:

| Setting | Default | Can it change after the build? |
|---|---|---|
| `source` | *(required)* | Yes, the data source is updated in place |
| `embedding` | `"titan-v2"` | **No** |
| `dimensions` | model default (1024 for titan-v2) | **No** |
| `vector_store` | `"s3_vectors"` | **No** |
| `chunking` | fixed, 300 tokens, 20% overlap | **No** |
| `role_arn` | kbforge creates a role | |
| `permissions_boundary` | none | |
| `region` | your notebook's region | |
| `description` | `""` | |
| `tags` | `{}` (`managed-by: kbforge` is always added) | |
| `sync` | `True` (first ingestion runs during `apply()`) | |

The settings marked **No** are fixed once the knowledge base exists. To try different ones, use a new name (see section 10).

The cells in this section only build Python objects so you can see what each option looks like. **They don't call AWS**, so run them freely. Your real definition goes in section 4.

### 3.1 Where your documents live: `source`

**S3**: pass an `s3://` path. Everything under the folder (prefix) is ingested. Leave the folder off to use the whole bucket.

Bedrock reads `.pdf .docx .doc .txt .md .html .csv .xls .xlsx` files up to 50 MB each. Other files are skipped, and the plan tells you how many.

In [ ]:
from kbforge import S3Source

docs = S3Source("s3://my-company-docs/hr/")   # a plain string works too: source="s3://my-company-docs/hr/"
print("bucket:", docs.bucket)
print("folder:", docs.prefix or "(whole bucket)")

**A website**: Bedrock's managed web crawler fetches the pages. A plain `"https://..."` string crawls that one site with the defaults. Use `WebSource` when you want to control the crawl. **Only crawl sites you own or are allowed to crawl.**

| Option | Default | Meaning |
|---|---|---|
| `urls` | *(required)* | One or more seed URLs to start from |
| `scope` | `"HOST_ONLY"` | `"HOST_ONLY"` stays on the same host; `"SUBDOMAINS"` also follows links to subdomains |
| `max_pages` | no limit | Stop after this many pages |
| `rate_limit` | `300` | Pages per minute per host. Keep it polite |
| `include` | `[]` | Regular expressions: only crawl URLs that match |
| `exclude` | `[]` | Regular expressions: skip URLs that match |

In [ ]:
from kbforge import WebSource

site = WebSource(
    ["https://docs.example.com", "https://help.example.com"],
    scope="SUBDOMAINS",
    max_pages=500,
    rate_limit=120,
    include=[r".*/handbook/.*"],
    exclude=[r".*\.pdf$"],
)
site.describe()

### 3.2 Embedding model: `embedding` and `dimensions`

The embedding model turns each chunk of text into a vector (a list of numbers) so that chunks with similar meaning end up close together. The same model embeds your questions, which is why it can't change later.

In [ ]:
from kbforge import EMBEDDING_MODELS

print(f"{'alias':<14}{'max input':>10}  {'dimensions':<18}notes")
for m in EMBEDDING_MODELS.values():
    dims = " / ".join(str(d) for d in m.dimensions)
    print(f"{m.alias:<14}{m.max_input_tokens:>10}  {dims:<18}{m.note}")

- **`titan-v2`** (default) suits most cases. It's the only one where you can pick `dimensions`: `256`, `512` or `1024`. Smaller vectors are cheaper to store and slightly less accurate.
- **`cohere-en`** / **`cohere-multi`** give strong retrieval quality, but read only the **first 512 tokens** of each chunk. Keep chunks at 512 tokens or less (the plan warns you if they aren't).
- A raw Bedrock model id also works if you give `dimensions` too, e.g. `embedding="amazon.titan-embed-text-v2:0", dimensions=1024`.

In [ ]:
from kbforge import KnowledgeBase

# Smaller, cheaper vectors with Titan v2. Displaying the object shows the design; it doesn't call AWS.
KnowledgeBase("demo-small-vectors", source="s3://my-company-docs/hr/", embedding="titan-v2", dimensions=512)

### 3.3 Vector store: `vector_store`

Where the vectors are kept and searched.

| Value | Good for | Cost |
|---|---|---|
| `"s3_vectors"` (default) | Most knowledge bases. Serverless, nothing to manage | Pay per GB stored and per query; close to nothing when idle |
| `"opensearch_serverless"` | Lower query latency, heavy traffic | **Minimum hourly charge even when idle** (about 2 OCUs). Needs `kbforge[opensearch]` (see section 1) |

`"opensearch"` and `"aoss"` are accepted as short spellings.

In [ ]:
KnowledgeBase("demo-opensearch", source="s3://my-company-docs/hr/", vector_store="opensearch_serverless")

### 3.4 Chunking: `chunking`

Documents are cut into chunks before embedding, and search returns chunks, not whole files. This choice affects answer quality more than anything else here. A *token* is roughly three-quarters of a word.

| Strategy | Use it when | Settings (default) |
|---|---|---|
| **`fixed`** (default) | You're not sure. A safe choice for most document sets | `max_tokens` (300, 1–8192), `overlap_percent` (20, 1–99) |
| **`hierarchical`** | Long, structured documents such as policies, manuals or contracts. Small chunks are searched, and the bigger *parent* chunk around each match is returned so answers keep their context | `parent_tokens` (1500, 1–8192), `child_tokens` (300, smaller than parent), `overlap_tokens` (60, smaller than child) |
| **`semantic`** | Topics don't line up with paragraphs. Splits where the meaning changes. Each sync makes extra model calls, so it's slower and costs more | `max_tokens` (300, 1–8192), `buffer_size` (0, 0 or 1), `breakpoint_threshold` (95, 50–99; higher means fewer, bigger chunks) |
| **`none`** | Files are already small (FAQ entries, short notes). Each file is one chunk | none |

There are three ways to write it: a name (`chunking="semantic"`) for the defaults, a helper (`Chunking.semantic(max_tokens=400)`) to change settings, or a dict, which is how it's written in a YAML/JSON file.

In [ ]:
from kbforge import Chunking

options = [
    Chunking.fixed(max_tokens=500, overlap_percent=10),
    Chunking.hierarchical(parent_tokens=2000, child_tokens=400, overlap_tokens=50),
    Chunking.semantic(max_tokens=400, buffer_size=1, breakpoint_threshold=90),
    Chunking.none(),
]
for c in options:
    print(c.describe())

In [ ]:
# The same knowledge base, chunking written three ways:
by_name   = KnowledgeBase("demo-chunking", source="s3://my-company-docs/hr/", chunking="hierarchical")
by_helper = KnowledgeBase("demo-chunking", source="s3://my-company-docs/hr/", chunking=Chunking.hierarchical())
by_dict   = KnowledgeBase(
    "demo-chunking",
    source="s3://my-company-docs/hr/",
    chunking={"strategy": "hierarchical", "parent_tokens": 1500, "child_tokens": 300, "overlap_tokens": 60},
)
by_helper

Bad values are caught **here**, before anything is built, with a message that says how to fix them:

In [ ]:
mistakes = [
    lambda: Chunking.hierarchical(parent_tokens=300, child_tokens=500),  # child must be smaller than parent
    lambda: Chunking.fixed(max_tokens=20_000),                           # above Bedrock's 8,192 limit
    lambda: Chunking.from_dict({"strategy": "semantic", "overlap_percent": 20}),  # belongs to fixed
]
for make in mistakes:
    try:
        make()
    except ValueError as err:
        print("✗", err)

### 3.5 Permissions: `role_arn` and `permissions_boundary`

Bedrock needs an IAM role to read your documents and write vectors.

- **Default:** kbforge creates one for you, named `AmazonBedrockExecutionRoleForKnowledgeBase_kbf-<name>`, with only the access this knowledge base needs.
- **`role_arn=`**: if your account doesn't let you create roles, ask an admin for one that trusts `bedrock.amazonaws.com` and pass its ARN. Its name must start with `AmazonBedrockExecutionRoleForKnowledgeBase_`, otherwise Bedrock's managed policies won't let you pass it. The plan checks the role's trust policy.
- **`permissions_boundary=`**: if your organisation requires a boundary on every new role, give the policy ARN and kbforge attaches it to the role it creates.

In [ ]:
KnowledgeBase(
    "demo-existing-role",
    source="s3://my-company-docs/hr/",
    role_arn="arn:aws:iam::123456789012:role/AmazonBedrockExecutionRoleForKnowledgeBase_hr",
)

### 3.6 Everything else: `region`, `description`, `tags`, `sync`

| Option | Default | Meaning |
|---|---|---|
| `region` | your notebook's region | Where to build. Keep it the same as your S3 bucket's region |
| `description` | `""` | Free text shown in the Bedrock console |
| `tags` | `{}` | Extra tags on every resource. `managed-by: kbforge` is always added; that's how `destroy()` knows what it's allowed to delete |
| `sync` | `True` | Set to `False` to build everything now and run the first ingestion later with `kb.sync()` |
| `session` | default boto3 session | A `boto3.Session(profile_name=...)` to use another AWS profile |

## 4. Describe your knowledge base

This is your real definition. **Change the name and the source**, uncomment any setting from section 3 you want to change, and run the cell. The card shows the design. It hasn't been checked against AWS yet.

In [ ]:
from kbforge import Chunking, KnowledgeBase, S3Source, WebSource

kb = KnowledgeBase(
    "hr-policies",                          # letters, digits, - and _ (max 100). Every AWS name is derived from it
    source="s3://my-company-docs/hr/",      # or "https://docs.example.com", or S3Source(...) / WebSource(...)

    # embedding="titan-v2",                 # titan-v2 | cohere-en | cohere-multi | titan-v1       (fixed after build)
    # dimensions=1024,                      # titan-v2 only: 256 | 512 | 1024                     (fixed after build)
    # vector_store="s3_vectors",            # s3_vectors | opensearch_serverless                   (fixed after build)
    # chunking=Chunking.fixed(max_tokens=300, overlap_percent=20),
    #                                       # or "hierarchical" | "semantic" | "none", or a dict (fixed after build)

    # role_arn="arn:aws:iam::123456789012:role/AmazonBedrockExecutionRoleForKnowledgeBase_hr",
    # permissions_boundary="arn:aws:iam::123456789012:policy/MyBoundary",

    # region="us-east-1",                   # default: the notebook's region
    # description="HR policy documents for the internal assistant",
    # tags={"team": "people-ops", "cost-center": "1234"},
    # sync=True,                            # False: build now, run kb.sync() later
)
kb

**Or load the definition from a file.** If your team keeps definitions in git, load a `.yaml`/`.yml` or `.json` file instead. Both formats take the same keys. See [`examples/kb.yaml`](https://github.com/utkarsh5026/bedrcok/blob/main/examples/kb.yaml) for every option with comments, and [`examples/README.md`](https://github.com/utkarsh5026/bedrcok/blob/main/examples/README.md) for the JSON version. Typos in key names are reported, not ignored.

`region=` overrides the region in the file, which is handy for building the same knowledge base somewhere else.

In [ ]:
# kb = KnowledgeBase.from_file("kb.yaml")                     # or "kb.json"
# kb = KnowledgeBase.from_file("kb.yaml", region="eu-west-1")  # same definition, another region

## 5. Review the plan

This checks your credentials, that the embedding model works for you, what's in the bucket and whether the vector store is reachable. Then it lists every AWS resource it would create. **Read-only.** Run it as often as you like.

- ✓ **ok**: fine.
- ⚠ **warning**: worth reading, doesn't block (e.g. some files will be skipped, chunks longer than the model reads).
- ✗ **blocker**: must be fixed before `apply()` will run. The message says how. See [Troubleshooting](#Troubleshooting).

In [ ]:
plan = kb.plan()
plan

The plan is also a Python object, which is useful in scripts:

In [ ]:
print("Ready to build:", plan.ready)
print("Anything to do:", plan.has_work)
for problem in plan.blockers:
    print("✗", problem)

## 6. Build it

Creates everything and runs the first sync (unless `sync=False`), with live progress. A first build usually takes a few minutes, mostly the sync.

If something fails, fix it and **run this cell again**: finished steps are detected and skipped. If the plan has blockers, nothing is changed and the plan is shown again instead.

In [ ]:
result = kb.apply()
result

## 7. Try it

`ask()` shows the chunks that come back for a question, best match first, with a score and the document each came from. It's retrieval only (no LLM writes an answer), which is the quickest way to judge whether your chunking and embedding choices work for your documents.

`top_k` sets how many chunks come back (default 5).

In [ ]:
answers = kb.ask("How many days of parental leave do we get?", top_k=5)
answers

**Using the results in code.** `answers.texts` is the list of chunk texts, ready to put into a prompt for your own LLM call. `answers.results` holds the raw Bedrock results, including each chunk's score and source.

In [ ]:
context = "\n\n---\n\n".join(answers.texts)
print(f"{len(answers.texts)} chunks, {len(context):,} characters of context\n")

for r in answers.results:
    location = r.get("location", {})
    source = location.get("s3Location") or location.get("webLocation") or {}
    print(f"{r.get('score', 0):.3f}  {source.get('uri') or source.get('url')}")

## 8. Keep it up to date

After adding, changing or deleting documents in the source, re-sync. Only the changes are processed. `status()` shows the ids, the model, the vector store and how the last sync went (files added, updated, failed).

In [ ]:
kb.sync()

In [ ]:
kb.status()

You'll need the knowledge base id to use it from other code (Bedrock's `retrieve` or `retrieve_and_generate` APIs, Bedrock Agents, LangChain and so on):

In [ ]:
status = kb.status()
print("Knowledge base ID:", status.kb_id)
print("Data source ID:   ", status.ds_id)
print("Region:           ", status.region)

## 9. Share the definition

Save the definition so your team can rebuild, sync or delete the same knowledge base. The extension picks the format.

In [ ]:
kb.save("kb.yaml")
# kb.save("kb.json")
print(kb.to_yaml())

Anyone can then use it from a terminal, or in CI, with the `kbforge` command:

```bash
kbforge init kb.yaml            # write a commented starter file (kb.json for JSON)
kbforge plan kb.yaml            # read-only checks + what would be built (exit code 2 if blocked)
kbforge apply kb.yaml           # build it (asks first; -y to skip the question)
kbforge sync kb.yaml            # re-ingest after documents change
kbforge ask kb.yaml "question"  # try a query (-k 10 for more results)
kbforge status kb.yaml
kbforge destroy kb.yaml         # delete it (asks first)
kbforge models                  # list supported embedding models
```

Every command takes `--region` to override the region in the file.

## 10. Compare settings

Chunking, embedding model and vector store can't change once a knowledge base exists. To find out which settings work best for your documents, build a second knowledge base **with a new name** from the same source, ask both the same questions, then destroy the one you don't keep.

The `apply()` line is commented out so running the whole notebook doesn't build a second knowledge base by accident.

In [ ]:
kb_b = KnowledgeBase(
    "hr-policies-hier",
    source=kb.spec.source,
    embedding=kb.spec.embedding,
    chunking=Chunking.hierarchical(parent_tokens=1500, child_tokens=300),
)
kb_b.plan()
# kb_b.apply()

In [ ]:
# Once kb_b is built, compare the two on questions you know the answer to:
# for question in ["How many days of parental leave do we get?", "Who approves remote work?"]:
#     display(kb.ask(question, top_k=3), kb_b.ask(question, top_k=3))
#
# Then delete the one you don't keep:
# kb_b.destroy(confirm=True)

## 11. Clean up

`destroy()` only *shows* what would be deleted. Add `confirm=True` to delete.

- Your **source documents are never touched**.
- Only resources tagged `managed-by: kbforge` are deleted. `force=True` also deletes same-named resources kbforge didn't create; use it with care.

In [ ]:
kb.destroy()

In [ ]:
# kb.destroy(confirm=True)

## Troubleshooting

| You see | What to do |
|---|---|
| **AWS credentials**: couldn't call STS | The notebook has no AWS credentials. On SageMaker, check the notebook's execution role; locally, run `aws configure` or pass `session=boto3.Session(profile_name=...)` |
| **Embedding model**: your role can't invoke this model | Enable the model in the Bedrock console under *Model access*, and make sure your role allows `bedrock:InvokeModel` |
| **Embedding model**: isn't available in this region | Use another model, or set `region=` to a region that has it |
| **Source bucket** doesn't exist / can't read it | Check the `s3://` path and that the bucket is in the same region as the knowledge base |
| **Source documents**: no files found | The folder is empty or the prefix is wrong. Folder names are case-sensitive |
| **Chunk size** warning | Your chunks are longer than the model reads (Cohere: 512 tokens). Lower `max_tokens` / `child_tokens`, or use `titan-v2` |
| **S3 Vectors**: your role lacks `s3vectors:*` | Add the permissions in `docs/notebook-role-policy.json` to your notebook role, or use `vector_store="opensearch_serverless"` |
| **OpenSearch Serverless**: needs the OpenSearch client | `%pip install "kbforge[all] @ git+..."`, then restart the kernel |
| **Your IAM role** doesn't trust bedrock.amazonaws.com | Ask the role's owner to add `bedrock.amazonaws.com` to its trust policy |
| A knowledge base with this name already exists but uses a different model / store | Those can't change in place. Pick a new name, or `destroy(confirm=True)` the old one first |
| `apply()` stopped halfway | Fix the reported cause and run `kb.apply()` again; finished steps are skipped |
| The sync finished but some files failed | `kb.status()` shows the counts. Usually unsupported or over-50 MB files |

---
Prefer a form? Run `kbforge.wizard()` in a new cell. It walks through the same choices with clickable cards and builds the knowledge base for you.